In [79]:
import os
%pwd

'c:\\Users\\HP\\Desktop\\coding\\kidney-disease-classification'

In [ ]:
os.chdir('../')
%pwd

In [80]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/Ravi-Katiyar04/kidney-disease-classification.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="Ravi-Katiyar04"
os.environ["MLFLOW_TRACKING_PASSWORD"]="40f60a5b1e6c3038bcb89f072b6ed6e1d691791c"

In [81]:
import tensorflow as tf

In [82]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [83]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [84]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [85]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/CT-KIDNEY-DATASET",
            mlflow_uri="https://dagshub.com/Ravi-Katiyar04/kidney-disease-classification.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [86]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from mlflow.models import infer_signature
from urllib.parse import urlparse

In [87]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config
        self.valid_generator = None
        self.model = None
        self.score = None

    def _valid_generator(self):
        datagenerator_kwargs = dict(
            rescale=1.0 / 255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear",
            class_mode="categorical"
        )

        valid_datagenerator = (
            tf.keras.preprocessing.image.ImageDataGenerator(
                **datagenerator_kwargs
            )
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)

        self._valid_generator()

        self.score = self.model.evaluate(
            self.valid_generator,
            verbose=1
        )

        self.save_score()

    def save_score(self):
        scores = {
            "loss": float(self.score[0]),
            "accuracy": float(self.score[1])
        }

        save_json(
            path=Path("scores.json"),
            data=scores
        )

    def log_into_mlflow(self):
        import mlflow
        from mlflow.models import infer_signature

        # Configure the DagsHub MLflow tracking server
        tracking_uri = self.config.mlflow_uri
        mlflow.set_tracking_uri(tracking_uri)

        # Use a valid experiment name instead of a stale experiment ID
        experiment_name = "Kidney-Disease-Classification"

        experiment = mlflow.get_experiment_by_name(experiment_name)

        if experiment is None:
            experiment_id = mlflow.create_experiment(experiment_name)
        else:
            experiment_id = experiment.experiment_id

        if self.model is None or self.score is None:
            raise RuntimeError(
             "Run evaluation() before log_into_mlflow()."
            )

        # Get sample validation images
        sample_images, _ = next(iter(self.valid_generator))
        sample_images = sample_images[:5]

        # Infer the model signature
        sample_predictions = self.model.predict(
            sample_images,
            verbose=0
        )

        signature = infer_signature(
            sample_images,
            sample_predictions
        )

        # Start the run in the explicitly selected experiment
        with mlflow.start_run(experiment_id=experiment_id):
            mlflow.log_params(self.config.all_params)

            mlflow.log_metrics({
                "loss": float(self.score[0]),
                "accuracy": float(self.score[1])
            })

            mlflow.keras.log_model(
                model=self.model,
                name="model",
                signature=signature,
                registered_model_name="VGG16Model"
            )

In [88]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
   raise e

Found 3732 images belonging to 4 classes.
234/234 ━━━━━━━━━━━━━━━━━━━━ 95s 385ms/step - accuracy: 0.2980 - loss: 7.0161


Registered model 'VGG16Model' already exists. Creating a new version of this model...
2026/10/18 22:22:57 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VGG16Model, version 5
Created version '5' of model 'VGG16Model'.


🏃 View run auspicious-wolf-312 at: https://dagshub.com/Ravi-Katiyar04/kidney-disease-classification.mlflow/#/experiments/1/runs/1b383067c34844518ec07da3a47cad47
🧪 View experiment at: https://dagshub.com/Ravi-Katiyar04/kidney-disease-classification.mlflow/#/experiments/1
